# Export tiles as Xarray

In [ ]:
import ee
import xarray as xr
import numpy as np
import dask.array as da
import zarr

import odc.geo.xr  # noqa: F401

In [ ]:
# Authenticate and initialize
# ee.Authenticate()
ee.Initialize(opt_url="https://earthengine-highvolume.googleapis.com")

In [ ]:
dataset = "ACA/reef_habitat/v2_0"
ic = ee.ImageCollection(ee.Image(dataset))

In [ ]:
# Region of interest. Eventually, we need to do all of -180 to 180 and -32 to 32
left = 140
bottom = -26
right = 180
top = 0

# Close to full resolution is 0.00005
res = 0.00005
transform = [res, 0, left, 0, -res, top]

ds = xr.open_dataset(
    ic,
    engine='ee',
    geometry=[left, bottom, right, top],
    projection=ee.Projection(
        crs="epsg:4326", transform=transform
    ),
    chunks={"time": 1, "lon": 10000, "lat": 10000},
).squeeze().drop_vars("time")

# Clean up just the reef mask for now
reef_mask = ds.reef_mask.astype("uint8")
reef_mask = reef_mask.transpose("lat", "lon")  # Transposed because GEE
reef_mask.odc.nodata = 0

reef_mask

In [ ]:
# Write direct to Zarr
zarr_path = "reef_mask.zarr"

reef_mask.to_zarr(
    zarr_path,
    mode="w",
    compute=True,
    consolidated=False,
    write_empty_chunks=False,
)

In [ ]:
# Correctly ordered coordinate arrays
lon = np.arange(-180, 180, 0.00005)
lat = np.arange(-32, 32, 0.00005)[::-1]  # descending lat to match geotiff convention (top to bottom)

# Create dataset with proper axis order: (lat, lon)
world_ds = xr.Dataset(
    {
        "reef_mask": (
            ["lat", "lon"],
            da.full(
                shape=(len(lat), len(lon)),
                fill_value=0,
                chunks=(1000, 1000),
                dtype="uint8",
            ),
        )
    },
    coords={
        "lat": lat,
        "lon": lon,
    },
)

world_ds = world_ds.odc.assign_crs("EPSG:4326")

# Write Zarr
world_ds.to_zarr(
    "aca.zarr",
    mode="w",
    compute=False,
    consolidated=False,
    encoding={
        "reef_mask": {"dtype": "uint8", "_FillValue": 0},
    },
    write_empty_chunks=False,
)


In [ ]:
def get_index_bounds(template, chunk):
    """Given a full template and a chunk, return index slices for lat and lon."""

    # Latitude is descending, so reverse for searchsorted
    reversed_lat = template.lat.values[::-1]

    lat_start = len(reversed_lat) - np.searchsorted(reversed_lat, chunk.lat.values[0], side="right")
    lat_end   = len(reversed_lat) - np.searchsorted(reversed_lat, chunk.lat.values[-1], side="left") + 1

    lon_start = np.searchsorted(template.lon.values, chunk.lon.values[0], side="left")
    lon_end   = np.searchsorted(template.lon.values, chunk.lon.values[-1], side="right") + 1

    return {
        "lat": slice(lat_start, lat_end),
        "lon": slice(lon_start, lon_end),
    }

# Open the existing Zarr store and array
store = zarr.open("aca.zarr")
reef_array = store["reef_mask"]  # This is now a zarr.Array

# Write a region into the Zarr store
reef_mask.to_zarr(
    "aca.zarr",
    region=get_index_bounds(world_ds, reef_mask),
    consolidated=False,
    append_dim=None
)

In [ ]:
# Open the Zarr store for the region
zds = xr.open_zarr(
    "aca.zarr",
    consolidated=False,
    chunks={"lat": 1000, "lon": 1000},
).squeeze()

region = zds.sel(
    lat=slice(top, bottom),
    lon=slice(left, right),
)

region = region.transpose("lat", "lon")  # Transposed again!

region = region.odc.assign_crs("epsg:4326")
region.reef_mask.odc.write_cog("test_from_zarr.tif", overwrite=True)